# Employee Attrition Analysis and Prediction Using Explainable AI
### Master of Computer Applications (MCA) Major Project

This exploratory notebook documents the data science and machine learning research lifecycle for the project:
1. **Dataset Ingestion & Validation** (IBM HR Analytics benchmark dataset)
2. **Exploratory Data Analysis (EDA)** on voluntary turnover distributions
3. **Preprocessing & ColumnTransformer Pipelines** (handling categorical encoding and scaling)
4. **Class Imbalance Handling & Multi-Model Benchmark** (Logistic Regression, Decision Tree, Random Forest, XGBoost)
5. **Model Evaluation & Diagnostic Curves** (Accuracy, Precision, Recall, F1, ROC-AUC)
6. **Explainable AI with SHAP** (Global feature importance & local Waterfall attributions)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from pathlib import Path

# 1. Load the benchmark dataset
dataset_path = Path("../data/employee_attrition.csv")
df = pd.read_csv(dataset_path)
print("Dataset Shape:", df.shape)
df.head()

In [2]:
# 2. Target Variable Analysis
attrition_counts = df['Attrition'].value_counts()
print("Attrition Counts:")
print(attrition_counts)
print("\nAttrition Rate (%):")
print(df['Attrition'].value_counts(normalize=True) * 100)

In [3]:
# 3. Feature Cleaning and Train-Test Split
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Exclude identifiers and zero-variance columns
drop_cols = ['EmployeeNumber', 'EmployeeCount', 'Over18', 'StandardHours']
df_clean = df.drop(columns=[c for c in drop_cols if c in df.columns])

X = df_clean.drop(columns=['Attrition'])
y = df_clean['Attrition'].map({'No': 0, 'Yes': 1})

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ]
)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)
print("Transformed features matrix:", X_train.shape)

In [4]:
# 4. Multi-Model Benchmark
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Balanced Logistic Regression
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:, 1]

print("=== Logistic Regression Performance ===")
print(classification_report(y_test, y_pred_lr, zero_division=0))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob_lr), 4))

# Random Forest Classifier
rf = RandomForestClassifier(n_estimators=150, max_depth=8, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print("\n=== Random Forest Performance ===")
print(classification_report(y_test, y_pred_rf, zero_division=0))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob_rf), 4))

In [5]:
# 5. Explainable AI with SHAP
feature_names = preprocessor.get_feature_names_out()
explainer = shap.Explainer(lr, X_train)
shap_values = explainer(X_test[:100])
print("SHAP calculation successful on test sample.")